In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
sellin = pd.read_csv("datasets/sell-in.csv", sep='\t')
productos = pd.read_csv("datasets/tb_productos.csv", sep='\t')
stocks = pd.read_csv("datasets/tb_stocks.csv", sep='\t')

In [3]:
# Verificación inicial
print(f"Sell-In: {sellin.shape[0]} filas y {sellin.shape[1]} columnas")
print(f"Productos: {productos.shape[0]} filas y {productos.shape[1]} columnas")
print(f"Stocks: {stocks.shape[0]} filas y {stocks.shape[1]} columnas")

Sell-In: 1217022 filas y 7 columnas
Productos: 1262 filas y 6 columnas
Stocks: 13691 filas y 3 columnas


In [4]:
# 3. MERGE INICIAL
df = sellin.merge(productos, on="product_id", how="left")
df = df.merge(stocks, on=["product_id", "periodo"], how="left")
print(f"Ventas-Productos-Stocks: {df.shape[0]} filas y {df.shape[1]} columnas")

Ventas-Productos-Stocks: 1234547 filas y 13 columnas


In [5]:
productos_clean = productos.drop_duplicates(subset=['product_id'], keep='first')
print(productos_clean.shape)

(1251, 6)


In [6]:
df = sellin.merge(productos_clean, on="product_id", how="left")
df = df.merge(stocks, on=["product_id", "periodo"], how="left")
print(sellin.shape)
print(df.shape)

(1217022, 7)
(1217022, 13)


In [7]:
df['periodo_dt'] = pd.to_datetime(df['periodo'].astype(str), format='%Y%m')

# Armado del Dataset

In [8]:
# Convertimos periodo a datetime
df["periodo_dt"] = pd.to_datetime(df["periodo"].astype(str), format="%Y%m")

# Determinar vida útil de cada producto (primer y último periodo)
vida_producto = df.groupby("product_id")["periodo_dt"].agg(["min", "max"]).reset_index()

# Expandimos cada producto con todos los periodos de su vida útil
periodos_producto = []
for _, row in vida_producto.iterrows():
    periodos = pd.date_range(start=row["min"], end=row["max"], freq="MS")
    for p in periodos:
        periodos_producto.append((p, row["product_id"]))

df_producto_periodo = pd.DataFrame(periodos_producto, columns=["periodo_dt", "product_id"])



# Agregar columna periodo en formato AAAAMM
df_producto_periodo["periodo"] = df_producto_periodo["periodo_dt"].dt.strftime("%Y%m").astype(int)

df_producto_periodo.drop(columns=['periodo_dt'],inplace=True)
# Ordenar por periodo_dt (ascendente) y luego por product_id
df_producto_periodo = df_producto_periodo.sort_values(by=["product_id", "periodo"], ascending=True).reset_index(drop=True)



###########
toneladas_vendidas = df.copy()
# Agregar los datos por periodo y product_id para obtener la serie temporal
# Sumamos tn, cust_request_qty y cust_request_tn por periodo y product_id
toneladas_vendidas = df.groupby(['periodo', 'product_id']).agg({
    'tn': 'sum',
    'cust_request_qty': 'sum',
    'cust_request_tn': 'sum'
}).reset_index()

# Paso 5: Unir con las toneladas efectivamente vendidas (tn)
df_merge = df_producto_periodo.merge(toneladas_vendidas[["periodo", "product_id", "tn"]],
                             on=["periodo", "product_id"],
                             how="left")

print(df_merge.shape)
df_merge["tn"] = df_merge["tn"].fillna(0)

(11633, 3)


# Prophet

In [ ]:
df_merge.head()

,product_id,periodo,tn
0,20001,201701,934.77222
1,20001,201702,798.01620
2,20001,201703,1303.35771
3,20001,201704,1069.96130
4,20001,201705,1502.20132


# Prophet

In [16]:
import pandas as pd
from prophet import Prophet

df = df_merge.copy()

# Asegurar que el campo periodo esté en formato datetime (Prophet espera columna 'ds' de fechas)
df['ds'] = pd.to_datetime(df['periodo'].astype(str), format='%Y%m')
df = df.rename(columns={'tn': 'y'})

# 📤 2. Filtrar solo las columnas necesarias
df = df[['product_id', 'ds', 'y']]


productos_ok = pd.read_csv("./datasets/product_id_apredecir201912.csv", sep="\t")



# Para guardar las predicciones
resultados = []

# ---------------------
#  Loop por producto
# ---------------------
for product_id in productos_ok['product_id'].unique():
    df_prod = df[df['product_id'] == product_id].sort_values('ds')

    if len(df_prod) < 6:
        continue  # opcional: saltear series demasiado cortas

    # Entrenar Prophet
    model = Prophet(yearly_seasonality=True)
    model.fit(df_prod[['ds', 'y']])

    # Predecir mes +2
    ultima_fecha = df_prod['ds'].max()
    future = model.make_future_dataframe(periods=2, freq='MS')
    future = future[future['ds'] > ultima_fecha]  # solo fechas futuras
    pred = model.predict(future)

    # Obtener solo la predicción de mes+2
    pred_mes2 = pred.tail(1)

    resultados.append({
        'product_id': product_id,
        'fecha_predicha': pred_mes2['ds'].values[0],
        'yhat': pred_mes2['yhat'].values[0],
        'yhat_lower': pred_mes2['yhat_lower'].values[0],
        'yhat_upper': pred_mes2['yhat_upper'].values[0]
    })

# ---------------------
# 📊 Resultados finales
# ---------------------
df_predicciones = pd.DataFrame(resultados)
print(df_predicciones.head())


Se truncaron las últimas líneas 5000 del resultado de transmisión.
DEBUG:cmdstanpy:input tempfile: /tmp/tmp7g3d7xbi/nvgelewx.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['/usr/local/lib/python3.11/dist-packages/prophet/stan_model/prophet_model.bin', 'random', 'seed=16233', 'data', 'file=/tmp/tmp7g3d7xbi/rh8phzck.json', 'init=/tmp/tmp7g3d7xbi/nvgelewx.json', 'output', 'file=/tmp/tmp7g3d7xbi/prophet_modelg03pye24/prophet_model-20250719004533.csv', 'method=optimize', 'algorithm=newton', 'iter=10000']
00:45:33 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing
00:45:33 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing
INFO:prophet:Disabling weekly seasonality. Run prophet with weekly_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
INFO:prophet:n_changepoints greater

   product_id fecha_predicha         yhat   yhat_lower   yhat_upper
0       20001     2018-04-01  1306.099844  1293.928398  1319.832485
1       20002     2018-04-01   962.326211   957.162378   967.432622
2       20003     2018-04-01   975.546005   975.545998   975.546013
3       20004     2018-04-01   419.521176   370.814602   464.842753
4       20005     2018-04-01 -1263.172198 -1266.981989 -1259.155763


In [ ]:
!pip install prophet

Defaulting to user installation because normal site-packages is not writeable


In [17]:
df_predicciones

,product_id,fecha_predicha,yhat,yhat_lower,yhat_upper
0,20001,2018-04-01,1306.099844,1293.928398,1319.832485
1,20002,2018-04-01,962.326211,957.162378,967.432622
2,20003,2018-04-01,975.546005,975.545998,975.546013
3,20004,2018-04-01,419.521176,370.814602,464.842753
4,20005,2018-04-01,-1263.172198,-1266.981989,-1259.155763
...,...,...,...,...,...
531,21200,2018-04-01,0.126195,0.123605,0.128971
532,21202,2018-04-01,0.198373,0.198373,0.198373
533,21218,2018-04-01,-0.233607,-0.233607,-0.233607
534,21222,2018-04-01,0.290158,0.268930,0.309716


In [4]:
def promedio_12_meses_780p():

    df = pd.read_csv("./datasets/periodo_x_producto_con_target.csv", sep=',', encoding='utf-8')
    df = df[df['periodo'] >= 201901]  # Filtrar desde 201901

    productos_ok = pd.read_csv("../../data/raw/product_id_apredecir201912.csv", sep="\t")

    df = df.merge(productos_ok, on='product_id', how='inner')

    df = df.groupby('product_id').agg({'tn': 'mean'}).reset_index()

    return df

df_promedios = promedio_12_meses_780p()

In [19]:
df_predicciones = df_predicciones.merge(df_promedios, on='product_id', how='left')
df_predicciones

,product_id,fecha_predicha,yhat,yhat_lower,yhat_upper,tn
0,20001,2018-04-01,1306.099844,1293.928398,1319.832485,1454.732720
1,20002,2018-04-01,962.326211,957.162378,967.432622,1175.437142
2,20003,2018-04-01,975.546005,975.545998,975.546013,784.976407
3,20004,2018-04-01,419.521176,370.814602,464.842753,627.215328
4,20005,2018-04-01,-1263.172198,-1266.981989,-1259.155763,668.270104
...,...,...,...,...,...,...
531,21200,2018-04-01,0.126195,0.123605,0.128971,0.113957
532,21202,2018-04-01,0.198373,0.198373,0.198373,0.092505
533,21218,2018-04-01,-0.233607,-0.233607,-0.233607,0.055985
534,21222,2018-04-01,0.290158,0.268930,0.309716,0.059215


In [25]:
df_predicciones.loc[df_predicciones['yhat'] < 0, 'yhat'] = df_predicciones['tn']

In [27]:
df_pred = df_predicciones[['product_id','yhat']].copy()
df_pred.rename(columns={'yhat': 'tn'}, inplace=True)
df_pred[['product_id', 'tn']].to_csv("./datasets/prophet_v1.csv", index=False, sep=',', encoding='utf-8')

In [5]:
df_kaggle = pd.read_csv("./datasets/prophet_v1.csv", sep=',')
df_kaggle = df_kaggle.merge(df_promedios, on='product_id', how='outer')
df_kaggle

,product_id,tn_x,tn_y
0,20001,1306.099844,1454.732720
1,20002,962.326211,1175.437142
2,20003,975.546005,784.976407
3,20004,419.521176,627.215328
4,20005,668.270104,668.270104
...,...,...,...
775,21263,NaN,0.029993
776,21265,NaN,0.089541
777,21266,NaN,0.094659
778,21267,NaN,0.092835


In [6]:
df_kaggle.loc[df_kaggle['tn_x'].isna(), 'tn_x'] = df_kaggle['tn_y']
df_kaggle.rename(columns={'tn_x': 'tn'}, inplace=True)
df_kaggle[['product_id', 'tn']].to_csv("./datasets/prophet_v1_780.csv", index=False, sep=',', encoding='utf-8')